In [1]:
import pandas as pd
import torch
import anndata as ad
from scipy.spatial import Delaunay
import sys
script_dir='/fs/ess/PAS1475/yzhong/sf_project/codebase'
sys.path.append(script_dir)
from utils.graph_build import filter_edge, edge_index_from_delaunay, pyg_obj
import scgpt.tasks as sct

/fs/ess/PAS1475/yzhong/sf_project/conda_env/scgpt/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/fs/ess/PAS1475/yzhong/sf_project/conda_env/scgpt/lib/python3.10/site-packages/scgpt/model/model.py:21: UserWarning: flash_attn is not installed
  warnings.warn("flash_attn is not installed")
/fs/ess/PAS1475/yzhong/sf_project/conda_env/scgpt/lib/python3.10/site-packages/scgpt/model/multiomic_model.py:19: UserWarning: flash_attn is not installed
  warnings.warn("flash_attn is not installed")
/fs/ess/PAS1475/yzhong/sf_project/conda_env/scgpt/lib/python3.10/site-packages/torchtext/vocab/__init__.py:4: UserWarning: 
/!\ IMPORTANT WARNING ABOUT TORCHTEXT STATUS /!\ 
Torchtext is deprecated and the last released version will be 0.18 (this one). You can silence this warning by calling the following at the beginnig

In [2]:
file_path='/fs/ess/PAS1475/yzhong/sf_project/datasets/segmentation_noise'

In [3]:
sample_name_l=["Lung5_Rep1","Lung5_Rep2","Lung5_Rep3","Lung6",
               "Lung9_Rep1","Lung9_Rep2","Lung12","Lung13"]

for sample_name in sample_name_l:

    adata_dict = torch.load(file_path+f'/CosMx_Human_{sample_name}_noise_processed.pt')
    print(adata_dict)

    model_dir='/fs/ess/PAS1475/yzhong/sf_project/backup/ccv/datasets/scGPT_human'
    gene_col='index'
    adata_scgpt_graph_l = []
    threshold = 0.99

    for key, adata in adata_dict.items():
        # Node features
        adata = sct.embed_data(
                            adata,
                            model_dir,
                            gene_col=gene_col,
                            batch_size=512,
                            return_new_adata=False
                            )
        X_scgpt = adata.obsm["X_scGPT"]
        adata_scgpt_f = torch.tensor(X_scgpt, dtype=torch.float)

        # Build edges from Delaunay + filter
        tri = Delaunay(adata.obs[['new_x', 'new_y']])
        adata_edge = filter_edge(tri, threshold=threshold).simplices
        edge_index = edge_index_from_delaunay(adata_edge)  # must output [2, E]

        # Build pyg graph
        adata_scgpt_graph = pyg_obj(adata_scgpt_f, edge_index)

        # # Add labels
        adata_scgpt_graph.niche = torch.tensor(
            pd.Categorical(adata.obs['niche']).codes,
            dtype=torch.long
        )
        adata_scgpt_graph.cell_type = torch.tensor(
            pd.Categorical(adata.obs['cell_type']).codes,
            dtype=torch.long
        )
        adata_scgpt_graph.dataset = key

        print(f'{key}: pyG graph', adata_scgpt_graph)
        adata_scgpt_graph_l.append(adata_scgpt_graph)

    torch.save(adata_scgpt_graph_l, file_path+f'/CosMx_Human_{sample_name}_noise_processed_og_graph_scgpt_test.pt')

{'Lung5_Rep1_0.5': AnnData object with n_obs × n_vars = 90650 × 960
    obs: 'orig.ident', 'nCount_RNA', 'nFeature_RNA', 'raw_sdimx', 'raw_sdimy', 'raw_cell_ID', 'viz_sdimx', 'viz_sdimy', 'viz_cell_ID', 'cell_ID', 'fov', 'Area', 'AspectRatio', 'Width', 'Height', 'Mean.CD298', 'Max.CD298', 'Mean.G', 'Max.G', 'Mean.Y', 'Max.Y', 'Mean.R', 'Max.R', 'Mean.DAPI', 'Max.DAPI', 'dualfiles', 'Slide_name', 'tissue', 'Run_name', 'ISH.concentration', 'Dash', 'slide_ID_numeric', 'Run_Tissue_name', 'Panel', 'Diversity', 'totalcounts', 'log10totalcounts', 'background', 'remove_flagged_cells', 'patient', 'cell_type', 'niche', 'prop_tumor_in_100_neighbors', 'n_counts', 'leiden', 'new_x', 'new_y'
    var: 'feat_ID', 'remove_flagged_genes', 'n_cells'
    uns: 'log1p'
    layers: 'counts', 'Lung5_Rep1_1.0': AnnData object with n_obs × n_vars = 90650 × 960
    obs: 'orig.ident', 'nCount_RNA', 'nFeature_RNA', 'raw_sdimx', 'raw_sdimy', 'raw_cell_ID', 'viz_sdimx', 'viz_sdimy', 'viz_cell_ID', 'cell_ID', 'fov', 

/fs/ess/PAS1475/yzhong/sf_project/conda_env/scgpt/lib/python3.10/site-packages/scgpt/model/model.py:77: UserWarning: flash-attn is not installed, using pytorch transformer instead. Set use_fast_transformer=False to avoid this warning. Installing flash-attn is highly recommended.
  warnings.warn(
Embedding cells: 100%|██████████| 178/178 [00:33<00:00,  5.37it/s]
/fs/ess/PAS1475/yzhong/sf_project/conda_env/scgpt/lib/python3.10/site-packages/scgpt/tasks/cell_emb.py:279: ImplicitModificationWarning: Setting element `.obsm['X_scGPT']` of view, initializing view as actual.
  adata.obsm["X_scGPT"] = cell_embeddings


Lung5_Rep1_0.5: pyG graph Data(x=[90650, 512], edge_index=[2, 532888], niche=[90650], cell_type=[90650], dataset='Lung5_Rep1_0.5')
scGPT - INFO - match 958/960 genes in vocabulary of size 60697.


/fs/ess/PAS1475/yzhong/sf_project/conda_env/scgpt/lib/python3.10/site-packages/scgpt/model/model.py:77: UserWarning: flash-attn is not installed, using pytorch transformer instead. Set use_fast_transformer=False to avoid this warning. Installing flash-attn is highly recommended.
  warnings.warn(
Embedding cells: 100%|██████████| 178/178 [00:31<00:00,  5.58it/s]
/fs/ess/PAS1475/yzhong/sf_project/conda_env/scgpt/lib/python3.10/site-packages/scgpt/tasks/cell_emb.py:279: ImplicitModificationWarning: Setting element `.obsm['X_scGPT']` of view, initializing view as actual.
  adata.obsm["X_scGPT"] = cell_embeddings


Lung5_Rep1_1.0: pyG graph Data(x=[90650, 512], edge_index=[2, 533302], niche=[90650], cell_type=[90650], dataset='Lung5_Rep1_1.0')
scGPT - INFO - match 958/960 genes in vocabulary of size 60697.


...

Lung13_1.0: pyG graph Data(x=[76421, 512], edge_index=[2, 450696], niche=[76421], cell_type=[76421], dataset='Lung13_1.0')
scGPT - INFO - match 958/960 genes in vocabulary of size 60697.


/fs/ess/PAS1475/yzhong/sf_project/conda_env/scgpt/lib/python3.10/site-packages/scgpt/model/model.py:77: UserWarning: flash-attn is not installed, using pytorch transformer instead. Set use_fast_transformer=False to avoid this warning. Installing flash-attn is highly recommended.
  warnings.warn(
Embedding cells: 100%|██████████| 150/150 [00:32<00:00,  4.59it/s]
/fs/ess/PAS1475/yzhong/sf_project/conda_env/scgpt/lib/python3.10/site-packages/scgpt/tasks/cell_emb.py:279: ImplicitModificationWarning: Setting element `.obsm['X_scGPT']` of view, initializing view as actual.
  adata.obsm["X_scGPT"] = cell_embeddings


Lung13_2.0: pyG graph Data(x=[76421, 512], edge_index=[2, 452314], niche=[76421], cell_type=[76421], dataset='Lung13_2.0')
scGPT - INFO - match 958/960 genes in vocabulary of size 60697.


/fs/ess/PAS1475/yzhong/sf_project/conda_env/scgpt/lib/python3.10/site-packages/scgpt/model/model.py:77: UserWarning: flash-attn is not installed, using pytorch transformer instead. Set use_fast_transformer=False to avoid this warning. Installing flash-attn is highly recommended.
  warnings.warn(
Embedding cells: 100%|██████████| 150/150 [00:34<00:00,  4.37it/s]
/fs/ess/PAS1475/yzhong/sf_project/conda_env/scgpt/lib/python3.10/site-packages/scgpt/tasks/cell_emb.py:279: ImplicitModificationWarning: Setting element `.obsm['X_scGPT']` of view, initializing view as actual.
  adata.obsm["X_scGPT"] = cell_embeddings


Lung13_5.0: pyG graph Data(x=[76421, 512], edge_index=[2, 454164], niche=[76421], cell_type=[76421], dataset='Lung13_5.0')
